## NBA Team Performance and Playoff History (2015-2025)

# 2. Dataset Overview

**Dataset Name:** NBA Team Performance and Playoff History (2015-2025)

**Data Source:** Basketball Data DuckDB Database

**Time Period:** 2014-2015 through 2024-2025 NBA seasons

**Number of Observations:** 330 team-season records (30 NBA teams across 11 seasons)

**Number of Variables:** 10 columns in processed dataset 

Dataset Description:
This dataset contains historical NBA team performance and playoff information from the 2014-2015 through 2024-2025 seasons. The data was extracted from the Basketball Data Duck DB database using the team_seasons, bbref_team_stats, and games tables. Each observation represents one NBA team's performance during a specific season. The database has a lot of useful information about regular season games, all-star games, NBA Cup games, and playoff games. 

The dataset focuses on key components including regular-season conference standings, playoff qualification, playoff advancement, and historical team performance. These specific variables will help us investigate whether a team's recent success or lack of success is associated with the likelihood of its players being traded, especially those with larger contracts. For example, teams that consistently miss the playoffs may approach roster changes differetnly from teams that frequently advance deep into the postseason. 

The dataset also includes historical five-year performance measures to examine longer-term patterns rather than relying only on a team's performance in a single season. Individual seasons may be affected by injuries, roster changes, or other circumstances that do not necessarily reflect a team's long-term performance. Therefore, we included data from previous five seasons to provide a more comprehensive representation of each team's historical success. 

### 2A. Important Variables and Units
| Variable | Units / Values | Description |
|---|---|---|
| `season` | Year (2015–2025) | Represents the ending year of an NBA season. For example, 2015 represents the 2014–2015 season. |
| `team_id` | Unique identifier | Identifies each NBA team and allows team data to be merged across datasets. |
| `nba_abbrev` | Three-letter abbreviation | Represents the team's abbreviated name, such as GSW for Golden State Warriors. |
| `full_name` | Text | The complete name of the NBA team. |
| `conference` | East or West | Identifies which NBA conference the team belongs to. |
| `approx_conference_rank` | Integer (1–15) | Estimates a team's final regular-season conference standing based on winning percentage. A ranking of 1 represents the highest-ranked team in the conference. Identical records may result in tied rankings. |
| `made_playoffs` | Boolean (True/False) | Indicates whether the team qualified for the NBA playoffs during that season. |
| `furthest_playoff_round` | Integer (0–4) | Represents how far a team advanced in the playoffs. A value of 0 indicates that the team missed the playoffs, while 4 indicates reaching the NBA Finals. |
| `playoff_appearances_last_5` | Count (0–5) | Number of times a team qualified for the playoffs during the five seasons preceding the current season. |
| `avg_playoff_round_last_5` | Average (0–4) | Average playoff round reached during the previous five seasons, counting seasons in which the team missed the playoffs as 0. |

## 2B. Dataset Concerns and Limitations
**1. Tied Confernce Standings**

One limitation of our dataset is that regular-season conference rankings were calculated using team winning percentages rather than official NBA tiebreaking procedures. Across the 2015-2025 seasons, we identified 41 groups of teams with identical records in the same confernce, involving 91 team-season observations. Since we did not apply official tiebreakers, some teams share the same approximate conference ranking even though they had different final standings. This may introduce some uncertainty into our predictive model.

**2. Missing Five-Year Historical Data**

Our dataset covers the 2014-2015 through 2024-2025 NBA seasons. However, calculating five-year performance trends require infromation from the five seasons preceding each observation. Currently, the 2015-2019 observations have insufficient historical records, resulting in 150 missing values for each of our two five-year performance variables.

To address this issue, we plan to look through the DuckDB database and retrieve additional records from the 2009-2010 through 2013-2014 seasons. Therefore, we can use these values to calculate the missing historical trends while keeping our final dataset restricted to the 2014-2015 through 2024-2025 seasons.

**3. Team Performance**

Our dataset measures team-level performance rather than individual player statistics. Although team success may influence roster decisions, other important factors such as player contracts, injuries, age, and individual performance may also affect whether a player gets traded. Therefore, this datset will be combined with our group's other datasets to provide a more comprehensive view of potential trade predictors.

**4. Historical Performance Does Not Gurantee Future Outcomes**

Although we use previous seasons to identify patters in team success, historical performance may not always represent a team's curren stiatuion. Other factors like injuries, coaching changes, roster adjustments, and other unexpected circumstances can significantly affect team performance. Our five-year trends help provide additional context, but they cannot fully account for these changes. 

### 3. Loading the Raw Data
**3A. Loading Raw Data**

In [44]:
#Loading Librairies

import pandas as pd
import duckdb
from pathlib import Path

#Define our data directories
RAW = Path("data/00-raw")
PROCESSED = Path("data/02-processed")

#Connect to the NBA DuckDB database
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("""
    ATTACH 'https://basketball-data.billmill.org/nba/nba.duckdb'
    AS nba (READ_ONLY)
""")

print("Succesfully connected to the NBA database!")

Succesfully connected to the NBA database!


In [45]:
#Load NBA team information
team_seasons_raw = con.execute("""
    SELECT *
    FROM nba.team_seasons
    WHERE season BETWEEN 2015 AND 2025
        AND is_nba = TRUE
    ORDER BY season, team_id
""").df()

#Load regular-season statistics and playoff qualification
team_stats_raw = con.execute("""
    SELECT 
        season,
        team_id,
        made_playoffs,
        wins,
        losses,
        g
    FROM nba.bbref_team_stats
    WHERE season BETWEEN 2015 AND 2025
    ORDER BY season, team_id
""").df()

#Load playoff game infomration
playoff_games_raw = con.execute("""
    SELECT 
        season,
        playoff_round,
        home_team_id,
        away_team_id
    FROM nba.games
    WHERE season BETWEEN 2015 and 2025
        AND game_type = 'playoffs'
        AND playoff_round BETWEEN 1 AND 4
""").df()

#Display the size of each dataset
print("Team information:", team_seasons_raw.shape)
print("Team statistics:", team_stats_raw.shape)
print("Playoff games:", playoff_games_raw.shape)

team_seasons_raw.head()

Team information: (330, 8)
Team statistics: (330, 6)
Playoff games: (915, 4)


,season,team_id,nba_abbrev,full_name,is_nba,conference,division,cup_group
0,2015,1610612737,ATL,Atlanta Hawks,True,East,Southeast,None
1,2015,1610612738,BOS,Boston Celtics,True,East,Atlantic,None
2,2015,1610612739,CLE,Cleveland Cavaliers,True,East,Central,None
3,2015,1610612740,NOP,New Orleans Pelicans,True,West,Southwest,None
4,2015,1610612741,CHI,Chicago Bulls,True,East,Central,None


### 3B. Checking Dataset Tidiness

Before cleaning and combining the datasets, we examined their structure to determine whether they followed tdy data principles. The team_seasons and bbref_team_stats tables contain one observation per team per season, while the games table contains one observation per playoff game.

The team-level tables can be merged using season and team_id. However, the playoff games dataset needs to be transformed because each game contains both a home team an away team. We will reorganize this information to determine the furthest playoff round reached by each team during each season. 

We also check for duplicate observations and confirm that each table contains the expected identifying columns before combining the data. 

In [46]:
#Check for duplicate team-season observations
print("Duplicate team information:",
      team_seasons_raw.duplicated(["season", "team_id"]).sum())

print("Duplicate team statistics:",
      team_stats_raw.duplicated(["season", "team_id"]).sum())

#Check missing team identifiers 
print("\nMissing identifiers in team information:")
print(team_seasons_raw[["season", "team_id"]].isna().sum())

print ("\nMissing identifiers in team statistics:")
print(team_stats_raw[["season", "team_id"]].isna().sum())

#Check playoff game structure
print("\nMissing playoff game identifiers:")
print(
    playoff_games_raw[
        ["season", "home_team_id", "away_team_id"]
    ].isna().sum()
)

#Confirm the number of NBA teams per season
print ("\nNBA teams per season:")
print(team_seasons_raw.groupby("season")["team_id"].nunique())

Duplicate team information: 0
Duplicate team statistics: 0

Missing identifiers in team information:
season     0
team_id    0
dtype: int64

Missing identifiers in team statistics:
season     0
team_id    0
dtype: int64

Missing playoff game identifiers:
season          0
home_team_id    0
away_team_id    0
dtype: int64

NBA teams per season:
season
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
Name: team_id, dtype: int64


### Tidiness Findings:

After examing the three datasets, we found that the team_seasons and bbref_team_stats tables each contain 330 team-season observations with no duplicate team-season identifiers. Both datasets have complete season and team_id values, making them suitable for mergins. 

The playoff games dataset contains 915 game-level observations. We also confirmed that its season, home team, and away team identifiers have no missing values. However, because each playoff game contains two participating teams, this dataset must be reorganized before being merged with the team-level tables.

Additionally, the NBA team information dataset contains exactly 30 unique teams for each seas from 2015 through 2025. This confirms that our filtering preserved all 30 NBA teams across the eleven seasons. 

Overall, the team-level datasets are already tidy with respect to their team-season identifiers and unique ID's. While the playoff-game dataset requires additional transformation to match our final dataset's structure. 

### 3C. Dataset Size

The three datasets used in our analysis have different number of observations because they represent different types of information

**NBA Team Statistics:** 330 rows and 8 columns, representing 30 NBA teams across 11 seasons.

**Regular-Season Team Statistics:** 330 rows and 6 selected columns, containing team records and playoff qualification information

**Playoff Games:** 915 rows and 4 selected columns, reprsesenting individual playoff games from 2015 through 2025.

These datasets will be combined into a single team-level dataset containing one observation per NBA team per season. The expected final dataset will contain 330 rows, with additional variables describing playoff performance and five-year historical trends. 

### 3D. Missing Data Anlysis 

Before cleaning and combing the datasets, we examined how many missing values were present in each dataset. Identifying missing values is important because incomplete records could affect our analysis and the accuracy of our pedictive model.

We also investigate whether missing values occur randomly or follow patterns associated with particular teams, seasons, or variables. Understanding these patterns helps us determine whether missing values should be removed, replaced, or preserved. 

In [47]:
# Check missing values in each dataset
print("Missing values in NBA Team Information:")
print(team_seasons_raw.isna().sum())

print("\nMissing values in Regular-Season Team Statistics:")
print(team_stats_raw.isna().sum())

print("\nMissing values in Playoff Games:")
print(playoff_games_raw.isna().sum())

# Calculate total missing values
print("\nTotal missing values by dataset:")
print("Team Information:", team_seasons_raw.isna().sum().sum())
print("Team Statistics:", team_stats_raw.isna().sum().sum())
print("Playoff Games:", playoff_games_raw.isna().sum().sum())

Missing values in NBA Team Information:
season          0
team_id         0
nba_abbrev      0
full_name       0
is_nba          0
conference      0
division        0
cup_group     270
dtype: int64

Missing values in Regular-Season Team Statistics:
season           0
team_id          0
made_playoffs    0
wins             0
losses           0
g                0
dtype: int64

Missing values in Playoff Games:
season           0
playoff_round    0
home_team_id     0
away_team_id     0
dtype: int64

Total missing values by dataset:
Team Information: 270
Team Statistics: 0
Playoff Games: 0


In [48]:
# Check which seasons have missing NBA Cup group information
cup_missing_by_season = (
    team_seasons_raw
    .groupby("season")["cup_group"]
    .apply(lambda x: x.isna().sum())
)

print("Missing NBA Cup group values by season:")
print(cup_missing_by_season)

Missing NBA Cup group values by season:
season
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024     0
2025     0
Name: cup_group, dtype: int64


**Missing Data Findings:**

After examing the three datasets, we found that the NBA Team Information dataset contains 270 missing values, all occurring in the cup_group column. The Regular-Season Team Statistics and Playoff Games datasets contain no missing values in the columns we selected.

To investigate the missing values further, we grouped the missing cup_group values by season. We discovered that all 30 teams have missing NBA Cup group information through 2015 through 2023, while the 2024 and 2025 seasons have no missing values. 

This indicates that the missingness is systematic rather than random because the NBA Cup was introduced during the 2023-2024 season. Therefore, NBA Cup group information was not applicable to the ealrier seaons. 

Since cup_group is not relevant to the team-performance variables we are analyzing, we decided to exclude this column from our final processed dataset rather than filing the missing values or removing entire observations. This allows us to preserve all 330 team-season records. 

### 3E. Outliers and Suspicious Entries

Before cleaning the datasets, we examined potential outliers and suspicious entries that could affect our analysis. We focused on checking whether regular-season wins and losses, playoff rounds, and team-season records contained values outside their expected ranges. 

Because NBA teams can have unusually successful or unsuccessful seasons, extreme values do not necessarily indicate errors. Therefore, we investigate suspicious observations before deciding whether they should be removed or preserved. 

In [49]:
# Examine regular-season wins, losses, and games played
print("Regular-season statistics:")
print(team_stats_raw[["wins", "losses", "g"]].describe())

# Check for suspicious regular-season records
invalid_records = team_stats_raw[
    (team_stats_raw["wins"] < 0) |
    (team_stats_raw["losses"] < 0) |
    (team_stats_raw["wins"] + team_stats_raw["losses"] != team_stats_raw["g"])
]

print("\nSuspicious regular-season records:", len(invalid_records))

# Check for invalid playoff rounds
invalid_rounds = playoff_games_raw[
    ~playoff_games_raw["playoff_round"].between(1, 4)
]

print("Invalid playoff rounds:", len(invalid_rounds))

# Check for games where a team plays itself
invalid_games = playoff_games_raw[
    playoff_games_raw["home_team_id"] ==
    playoff_games_raw["away_team_id"]
]

print("Games with identical home and away teams:", len(invalid_games))

Regular-season statistics:
             wins      losses           g
count  330.000000  330.000000  330.000000
mean    40.027273   40.027273   80.054545
std     12.156122   11.974486    4.260163
min     10.000000    9.000000   64.000000
25%     32.000000   32.000000   82.000000
50%     42.000000   39.000000   82.000000
75%     49.000000   48.000000   82.000000
max     73.000000   72.000000   82.000000

Suspicious regular-season records: 0
Invalid playoff rounds: 0
Games with identical home and away teams: 0


### Outlier and Suspicious Entry Findings:

After examining the regular-season statistics, we found that teams recorded betweeen 10 and 73 wins, while the number of games played ranged from 64 to 82. Although these values vary significantly, they represent possible NBA season outcomes rather than necessarily indicating incorrect data. Some seasons were shortened due to circumstances such as the COVID-19 pandemic, which explains why certain teams played fewer than the standard 82 games. 

We also checked whether any teams had negative wins or losses, whether wins and losses added up to the totlal number of games played, and whether playoff rounds fell otuside the expected range of 1-4. Additionally, we checked whether any playoff games listed the same team as both the home and away team.

Our check identified **zero suspicisou regular-season records, zero invalid playoff rounds, and zero games with identical home and away teams.** Therfore, we decided to preserve the observations rather than remove any potential outliers, since the extreme value appear plausible and our validation checks did not identify any inconsistencies.

### 3F. Cleaning and Feature Engineering

Before cleaning and merging the datasets, we first investigate whether additional historical records are available in our database. Our final dataset focuses on the 2014-2015 through 2024-2025 NBA seasons, but calculating five-year performance trends for the earlier observations require data from previous seasons.

To address this, we check whether the database contains NBA team statistics and playoff information dating back to the 2009-2010 season. If these records are available and complete, we can use them to calculate our historical performance variables while keeping the final dataset restricted to the original eleven seaons. 

In [50]:
# Check historical NBA team statistics availability
historical_team_coverage = con.execute("""
    SELECT
        season,
        COUNT(DISTINCT team_id) AS team_count
    FROM nba.bbref_team_stats
    WHERE season BETWEEN 2010 AND 2025
    GROUP BY season
    ORDER BY season
""").df()

# Check historical playoff game availability
historical_playoff_coverage = con.execute("""
    SELECT
        season,
        COUNT(*) AS playoff_games,
        COUNT(DISTINCT home_team_id) AS home_teams
    FROM nba.games
    WHERE season BETWEEN 2010 AND 2025
      AND game_type = 'playoffs'
      AND playoff_round BETWEEN 1 AND 4
    GROUP BY season
    ORDER BY season
""").df()

print("Historical team statistics coverage:")
print(historical_team_coverage)

print("\nHistorical playoff game coverage:")
print(historical_playoff_coverage)

Historical team statistics coverage:
    season  team_count
0     2010          30
1     2011          30
2     2012          30
3     2013          30
4     2014          30
5     2015          30
6     2016          30
7     2017          30
8     2018          30
9     2019          30
10    2020          30
11    2021          30
12    2022          30
13    2023          30
14    2024          30
15    2025          30

Historical playoff game coverage:
    season  playoff_games  home_teams
0     2010             82          16
1     2011             81          16
2     2012             84          16
3     2013             85          16
4     2014             89          16
5     2015             81          16
6     2016             86          16
7     2017             79          16
8     2018             82          16
9     2019             82          16
10    2020             83          16
11    2021             85          16
12    2022             87          16
13   

In [51]:
# Load historical playoff games from 2010 through 2025
historical_playoff_games = con.execute("""
    SELECT
        season,
        playoff_round,
        home_team_id,
        away_team_id
    FROM nba.games
    WHERE season BETWEEN 2010 AND 2025
      AND game_type = 'playoffs'
      AND playoff_round BETWEEN 1 AND 4
""").df()

# Separate the home and away teams
historical_home = historical_playoff_games[
    ["season", "playoff_round", "home_team_id"]
].rename(columns={"home_team_id": "team_id"})

historical_away = historical_playoff_games[
    ["season", "playoff_round", "away_team_id"]
].rename(columns={"away_team_id": "team_id"})

# Combine all participating teams
historical_all_teams = pd.concat(
    [historical_home, historical_away],
    ignore_index=True
)

# Identify each team's furthest playoff round
historical_team_rounds = (
    historical_all_teams
    .groupby(["season", "team_id"])["playoff_round"]
    .max()
    .reset_index()
    .rename(columns={"playoff_round": "furthest_playoff_round"})
)

# Validate the number of playoff teams and Finals participants
historical_validation = (
    historical_team_rounds
    .groupby("season")
    .agg(
        playoff_teams=("team_id", "nunique"),
        finals_teams=("furthest_playoff_round",
                      lambda x: (x == 4).sum())
    )
    .reset_index()
)

historical_validation

,season,playoff_teams,finals_teams
0,2010,16,2
1,2011,16,2
2,2012,16,2
3,2013,16,2
4,2014,16,2
5,2015,16,2
6,2016,16,2
7,2017,16,2
8,2018,16,2
9,2019,16,2


### Historical Playoff Data Validation:

After examining the additional historical records, we confirmed that the DuckDB database contains playoff information for every NBA season from 2009-2010 through 2024-2025. Each season contains 16 playoff teams and two NBA Finals participants, matching the expected playoff structure. 

These results allow us to extend our historical calculations back to 2010 without adding additional observations to our final dataset. We will use the earlier seasons only to calculate the five-year performance for our original 2015-2025 observations

In [52]:
# Load historical NBA team information and playoff qualification
historical_team_stats = con.execute("""
    SELECT
        season,
        team_id,
        made_playoffs
    FROM nba.bbref_team_stats
    WHERE season BETWEEN 2010 AND 2025
    ORDER BY team_id, season
""").df()

# Merge historical team statistics with playoff rounds
historical_performance = historical_team_stats.merge(
    historical_team_rounds,
    on=["season", "team_id"],
    how="left",
    validate="one_to_one"
)

# Check for inconsistencies before filling missing playoff rounds
mismatches = historical_performance[
    historical_performance["made_playoffs"].notna()
    & (
        historical_performance["made_playoffs"]
        != historical_performance["furthest_playoff_round"].notna()
    )
]

print("Historical dataset shape:", historical_performance.shape)
print("Mismatched playoff records:", len(mismatches))
print("Missing playoff statuses:",
      historical_performance["made_playoffs"].isna().sum())

# Only fill non-playoff rounds after validation
if len(mismatches) == 0 and historical_performance["made_playoffs"].notna().all():
    historical_performance["furthest_playoff_round"] = (
        historical_performance["furthest_playoff_round"]
        .fillna(0)
        .astype(int)
    )
    print("Historical playoff rounds successfully cleaned.")
else:
    print("Validation failed. Review missing or mismatched records.")

historical_performance.head(10)

Historical dataset shape: (480, 4)
Mismatched playoff records: 0
Missing playoff statuses: 0
Historical playoff rounds successfully cleaned.


,season,team_id,made_playoffs,furthest_playoff_round
0,2010,1610612737,True,2
1,2011,1610612737,True,2
2,2012,1610612737,True,1
3,2013,1610612737,True,1
4,2014,1610612737,True,1
5,2015,1610612737,True,3
6,2016,1610612737,True,2
7,2017,1610612737,True,1
8,2018,1610612737,False,0
9,2019,1610612737,False,0


**Calculating Five-Year Performance Trends:**

To examine long-term team performance, we calculated two additional variables using playoff data from the previous five seasons. The first variable counts how many times a team qualified for the playoffs, while the second calculates the average playoff round reached during those five seasons. 

We used historical data beginning in 2010 so that all observations from 2015 through 2025 have five complete preceding seasons. For each team, we excluded the current season from the historical calculations to ensure that only previous performance contributed to these variables.

The additional historical records are used only for feature engineering and will not be included as additional observations in our final dataset. 

In [53]:
# Sort historical records by team and season
historical_performance = historical_performance.sort_values(
    ["team_id", "season"]
).copy()

# Calculate playoff appearances over the previous five seasons
historical_performance["playoff_appearances_last_5"] = (
    historical_performance
    .groupby("team_id")["made_playoffs"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=5).sum())
)

# Calculate average playoff round over the previous five seasons
historical_performance["avg_playoff_round_last_5"] = (
    historical_performance
    .groupby("team_id")["furthest_playoff_round"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=5).mean())
)

# Keep only the original 2015–2025 study period
historical_trends_final = historical_performance[
    historical_performance["season"].between(2015, 2025)
].copy()

# Verify our results
print("Dataset shape:", historical_trends_final.shape)

print("\nMissing five-year trends:")
print(
    historical_trends_final[
        ["playoff_appearances_last_5", "avg_playoff_round_last_5"]
    ].isna().sum()
)

# Display a sample of calculated trends
historical_trends_final.head(10)

Dataset shape: (330, 6)

Missing five-year trends:
playoff_appearances_last_5    0
avg_playoff_round_last_5      0
dtype: int64


,season,team_id,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
5,2015,1610612737,True,3,5.0,1.4
6,2016,1610612737,True,2,5.0,1.6
7,2017,1610612737,True,1,5.0,1.6
8,2018,1610612737,False,0,5.0,1.6
9,2019,1610612737,False,0,4.0,1.4
10,2020,1610612737,False,0,3.0,1.2
11,2021,1610612737,True,3,2.0,0.6
12,2022,1610612737,True,1,2.0,0.8
13,2023,1610612737,True,1,2.0,0.8
14,2024,1610612737,False,0,3.0,1.0


**Merging Team Performance Data:**

After calculating the five-year historical performance variables, we combined them with the NBA team information and regular-season statistics for the 2015-2025 seasons.

We merged the datasets using season and team_id, which identify each team during a specific NBA season. This allows us to include team names, conferences, playoff qualification, playoff advancement, and historical performance in one dataset.

We use a left merge to preserve all 330 team-season observations and check for missing values or duplicate records after merging.

In [54]:
# Select historical performance variables
historical_features = historical_trends_final[
    [
        "season",
        "team_id",
        "made_playoffs",
        "furthest_playoff_round",
        "playoff_appearances_last_5",
        "avg_playoff_round_last_5"
    ]
].copy()

# Merge team information with regular-season statistics
team_performance_final = team_seasons_raw.merge(
    team_stats_raw,
    on=["season", "team_id"],
    how="left",
    validate="one_to_one"
)

# Merge historical playoff features
team_performance_final = team_performance_final.merge(
    historical_features,
    on=["season", "team_id"],
    how="left",
    validate="one_to_one",
    suffixes=("_stats", "")
)

# Check dataset structure
print("Dataset shape:", team_performance_final.shape)

print("\nMissing values:")
print(team_performance_final.isna().sum())

print("\nDuplicate team-season records:",
      team_performance_final.duplicated(["season", "team_id"]).sum())

team_performance_final.head(10)

Dataset shape: (330, 16)

Missing values:
season                          0
team_id                         0
nba_abbrev                      0
full_name                       0
is_nba                          0
conference                      0
division                        0
cup_group                     270
made_playoffs_stats             0
wins                            0
losses                          0
g                               0
made_playoffs                   0
furthest_playoff_round          0
playoff_appearances_last_5      0
avg_playoff_round_last_5        0
dtype: int64

Duplicate team-season records: 0


,season,team_id,nba_abbrev,full_name,is_nba,conference,division,cup_group,made_playoffs_stats,wins,losses,g,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
0,2015,1610612737,ATL,Atlanta Hawks,True,East,Southeast,None,True,60,22,82,True,3,5.0,1.4
1,2015,1610612738,BOS,Boston Celtics,True,East,Atlantic,None,True,40,42,82,True,1,4.0,2.0
2,2015,1610612739,CLE,Cleveland Cavaliers,True,East,Central,None,True,53,29,82,True,4,1.0,0.4
3,2015,1610612740,NOP,New Orleans Pelicans,True,West,Southwest,None,True,45,37,82,True,1,1.0,0.2
4,2015,1610612741,CHI,Chicago Bulls,True,East,Central,None,True,50,32,82,True,2,5.0,1.6
5,2015,1610612742,DAL,Dallas Mavericks,True,West,Southwest,None,True,50,32,82,True,1,4.0,1.4
6,2015,1610612743,DEN,Denver Nuggets,True,West,Northwest,None,False,30,52,82,False,0,4.0,0.8
7,2015,1610612744,GSW,Golden State Warriors,True,West,Pacific,None,True,67,15,82,True,4,2.0,0.6
8,2015,1610612745,HOU,Houston Rockets,True,West,Southwest,None,True,56,26,82,True,3,2.0,0.4
9,2015,1610612746,LAC,Los Angeles Clippers,True,West,Pacific,None,True,56,26,82,True,2,3.0,1.0


**Calculating Regular-Season Confernce Rankings:**

After merging the datasets, we calculated each team's approximate regular-season confernce ranking using its winning percentage. We first divided the number of games won by the total number of games played, then ranked teams within their respective conferences for each season.

Because multiple teams can finish with identical records, some teams may receive the same ranking. We identified 41 groups of tied records across the 2015-2025 seasons during our earlier anlysis. Since official NBA tiebreaking procedures were not incorporated, these values represent approximate confernce standings rather than official seeds. 

We also compare playoff qulaification from the regular-season statisitcs table with our historical playoff records to verify that the two sources are consistent before removing redundant columns. 

In [55]:
# Compare playoff qualification between the two sources
playoff_mismatches = (
    team_performance_final["made_playoffs_stats"]
    != team_performance_final["made_playoffs"]
)

print("Mismatched playoff statuses:", playoff_mismatches.sum())

# Calculate regular-season winning percentage
team_performance_final["win_pct"] = (
    team_performance_final["wins"] /
    team_performance_final["g"]
)

# Calculate approximate conference standings
team_performance_final["approx_conference_rank"] = (
    team_performance_final
    .groupby(["season", "conference"])["win_pct"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Flag teams with identical records in the same conference
team_performance_final["tied_record"] = (
    team_performance_final.duplicated(
        ["season", "conference", "wins", "losses"],
        keep=False
    )
)

print("Team-season records involved in ties:",
      team_performance_final["tied_record"].sum())

print("\nInvalid conference rankings:",
      (~team_performance_final["approx_conference_rank"].between(1, 15)).sum())

# Display a sample of conference standings
team_performance_final[
    team_performance_final["season"] == 2015
][
    ["full_name", "conference", "wins", "losses",
     "approx_conference_rank", "tied_record"]
].sort_values(["conference", "approx_conference_rank"]).head(15)

Mismatched playoff statuses: 0
Team-season records involved in ties: 91

Invalid conference rankings: 0


,full_name,conference,wins,losses,approx_conference_rank,tied_record
0,Atlanta Hawks,East,60,22,1,False
2,Cleveland Cavaliers,East,53,29,2,False
4,Chicago Bulls,East,50,32,3,False
24,Toronto Raptors,East,49,33,4,False
27,Washington Wizards,East,46,36,5,False
12,Milwaukee Bucks,East,41,41,6,False
1,Boston Celtics,East,40,42,7,False
14,Brooklyn Nets,East,38,44,8,True
17,Indiana Pacers,East,38,44,8,True
11,Miami Heat,East,37,45,10,False


**Final Data Cleaning:**

After merging our team performance datasets and calculating the historical features, we selected the variables most relevant to our NBA trade predicition project.

We removed unnecessary columns, including NBA Cup groups, divisions, and raw win/loss totals, because these variables are either unrelated to our assigned features or were only needed for intermediate calculations.

We also removed the duplicate playoff qualification column after comparing the two sources. The final dataset retains team identifiers, approximate confernce rankings, plaoff qualification, playoff advancement, and five-year historical performance trends.

Our final dataset covers the 2014-2015 through 2024-2025 NBA seasons, with one observation per team per season.

In [56]:
# Confirm playoff qualification agrees across both sources
assert (
    team_performance_final["made_playoffs_stats"]
    == team_performance_final["made_playoffs"]
).all(), "Playoff qualification mismatch detected!"

# Select only the relevant variables
team_features_final = team_performance_final[
    [
        "season",
        "team_id",
        "nba_abbrev",
        "full_name",
        "conference",
        "approx_conference_rank",
        "made_playoffs",
        "furthest_playoff_round",
        "playoff_appearances_last_5",
        "avg_playoff_round_last_5"
    ]
].copy()

# Sort by season and team identifier
team_features_final = team_features_final.sort_values(
    ["season", "team_id"]
).reset_index(drop=True)

# Display the cleaned dataset
print("Final dataset shape:", team_features_final.shape)

print("\nMissing values:")
print(team_features_final.isna().sum())

print("\nDuplicate team-season records:",
      team_features_final.duplicated(["season", "team_id"]).sum())

team_features_final.head(10)

Final dataset shape: (330, 10)

Missing values:
season                        0
team_id                       0
nba_abbrev                    0
full_name                     0
conference                    0
approx_conference_rank        0
made_playoffs                 0
furthest_playoff_round        0
playoff_appearances_last_5    0
avg_playoff_round_last_5      0
dtype: int64

Duplicate team-season records: 0


,season,team_id,nba_abbrev,full_name,conference,approx_conference_rank,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
0,2015,1610612737,ATL,Atlanta Hawks,East,1,True,3,5.0,1.4
1,2015,1610612738,BOS,Boston Celtics,East,7,True,1,4.0,2.0
2,2015,1610612739,CLE,Cleveland Cavaliers,East,2,True,4,1.0,0.4
3,2015,1610612740,NOP,New Orleans Pelicans,West,8,True,1,1.0,0.2
4,2015,1610612741,CHI,Chicago Bulls,East,3,True,2,5.0,1.6
5,2015,1610612742,DAL,Dallas Mavericks,West,7,True,1,4.0,1.4
6,2015,1610612743,DEN,Denver Nuggets,West,12,False,0,4.0,0.8
7,2015,1610612744,GSW,Golden State Warriors,West,1,True,4,2.0,0.6
8,2015,1610612745,HOU,Houston Rockets,West,2,True,3,2.0,0.4
9,2015,1610612746,LAC,Los Angeles Clippers,West,2,True,2,3.0,1.0


**4. Final Dataset Validation**

After cleaning and merging the datsets, we performed additional validation checks to ensure that the final team-performance dataset contains consistent and reasonable values.

We checked for duplicate team-season records, missing values, invalid conference rankings, and playoff rounds outside their expected ranges. We also verfified that the five-year historical performance variables fall within their expected ranges.

These checks help ensure that our processed dataset is suitable for combining with the other datasets used in our NBA trade prediction project.

In [57]:
# Verify the final dataset dimensions
print("Final dataset shape:", team_features_final.shape)

# Check for duplicate team-season records
print("Duplicate team-season records:",
      team_features_final.duplicated(["season", "team_id"]).sum())

# Check for missing values
print("Total missing values:",
      team_features_final.isna().sum().sum())

# Validate conference rankings and playoff rounds
print("Invalid conference rankings:",
      (~team_features_final["approx_conference_rank"].between(1, 15)).sum())

print("Invalid playoff rounds:",
      (~team_features_final["furthest_playoff_round"].between(0, 4)).sum())

# Validate five-year historical trends
print("Invalid five-year playoff appearances:",
      (~team_features_final["playoff_appearances_last_5"].between(0, 5)).sum())

print("Invalid five-year average playoff rounds:",
      (~team_features_final["avg_playoff_round_last_5"].between(0, 4)).sum())

# Verify the number of teams in each season
print("\nTeam-season records by year:")
print(team_features_final.groupby("season")["team_id"].nunique())

Final dataset shape: (330, 10)
Duplicate team-season records: 0
Total missing values: 0
Invalid conference rankings: 0
Invalid playoff rounds: 0
Invalid five-year playoff appearances: 0
Invalid five-year average playoff rounds: 0

Team-season records by year:
season
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
Name: team_id, dtype: int64


In [58]:
# Create processed-data directory if needed
PROCESSED.mkdir(parents=True, exist_ok=True)

# Define the output filename
output_path = PROCESSED / "jai_team_performance_2015_2025.csv"

# Save the updated cleaned dataset
team_features_final.to_csv(output_path, index=False)

print("Saved cleaned dataset to:", output_path)

Saved cleaned dataset to: data/02-processed/jai_team_performance_2015_2025.csv


**5. Final Processed Dataset**

After completing the cleaning and feature engineering process, our final dataset contains 330 observations and 10 variables, representing all 30 NBA teams across the 2014-2015 through 2024-2025 seasons.

Our final validation identified zero duplicate team-season records, zero missing values, and no values outside the expected ranges for playoff rounds, approximate conference rankings, or five-year performance measures.

We also successfully addressed the missing five-year historical data by retrieving additional NBA records from the 2009-2010 through 2013-2014 seasons. These earlier seasons were used only to calculate historical performance trends and were excluded from the final dataset.

The cleaned dataset was exported, where it can be combined with our group's player-level statistics and salary datasets for our NBA trade prdiction model.

**Remaining Limitation:** Our regular-season conference rankings remain approximate because teams with identical records were not separated usin official NBA tiebreaking procedures. Therefore, the approximate conference variable should be interpreted as an exact official conference seed. 